# Example 7: Fishing Intelligence with Global Fishing Watch

Access pre-computed fishing events, vessel identity, and gridded fishing effort from GFW.

Unlike NOAA and DMA, Global Fishing Watch does **not** provide raw AIS
position reports. Instead, it offers three higher-level datasets:

| Dataset | Description |
|---|---|
| **events** | Fishing activity, vessel encounters, port visits, loitering |
| **vessels** | Vessel identity extracted from event metadata (MMSI, name, flag) |
| **fishing_effort** | 4Wings gridded data — aggregated vessel-hours per spatial cell |

This example shows how to download and explore all three.

## Prerequisites

```bash
pip install neptune-ais[notebooks] gfw-api-python-client>=1.0
```

**GFW API token required.** Get a free token at
https://globalfishingwatch.org/our-apis/tokens and set it as an
environment variable:

```bash
export GFW_API_TOKEN="your-token-here"
```

Cells that hit the GFW API are guarded with `if token:` so the notebook
remains viewable even without a token.

## Setup

In [ ]:
import os

import polars as pl

# In Jupyter, the GFW adapter needs nest_asyncio to bridge its async client.
import nest_asyncio
nest_asyncio.apply()

from neptune_ais.api import Neptune

token = os.environ.get("GFW_API_TOKEN", "")
if not token:
    print("GFW_API_TOKEN not set — API cells will be skipped.")
    print("Set the env var and restart the kernel to enable them.")
else:
    print("GFW API token found.")

## Download GFW Data

Create a Neptune instance targeting GFW as the sole source. The
`api_keys` dict passes the token to the GFW adapter.

`.download()` fetches events and fishing effort for the requested date.
Because GFW doesn't provide positions, the positions step is
automatically skipped.

In [ ]:
if token:
    n = Neptune(
        "2024-06-15",
        sources=["gfw"],
        api_keys={"gfw": token},
        cache_dir="/tmp/neptune_demo",
    )
    partitions = n.download()
    print(f"Wrote {len(partitions)} partitions:")
    for p in partitions:
        print(f"  {p}")

## Fishing Events

GFW provides four event types through Neptune's canonical `events/v1` schema:

| Event type | GFW source | Description |
|---|---|---|
| `fishing` | Fishing Events API | Detected fishing activity |
| `encounter` | Encounters API | Two vessels in close proximity |
| `port_call` | Port Visits API | Vessel visiting a port |
| `loitering` | Loitering API | Sustained slow movement in open water |

These are **source-native** events — computed by GFW's own pipeline,
not derived from positions by Neptune's heuristic detectors
(see [Example 04](04_event_detection.ipynb) for the detector-based approach).

In [ ]:
if token:
    all_events = n.events().collect()
    print(f"{len(all_events)} total events")
    print()
    print("Events by type:")
    print(all_events.group_by("event_type").len().sort("len", descending=True))
    print()
    all_events.head(5)

### Filter by Event Type

Filter the collected events by type. GFW events are assigned
confidence 1.0 since they come pre-validated by GFW's pipeline.

In [ ]:
if token:
    fishing = all_events.filter(pl.col("event_type") == "fishing")
    print(f"{len(fishing)} fishing events")
    fishing.head(3)

In [ ]:
if token:
    encounters = all_events.filter(pl.col("event_type") == "encounter")
    print(f"{len(encounters)} encounter events")
    if len(encounters) > 0:
        # Encounters include the other vessel's MMSI
        print(encounters.select("event_id", "mmsi", "other_mmsi", "start_time", "lat", "lon").head(3))

## Vessel Identity

GFW event responses embed vessel metadata (name, flag, ship type).
Neptune extracts this into the `vessels/v1` dataset, giving you one
record per unique MMSI seen in the events.

In [ ]:
if token:
    vessels = n.vessels().collect()
    print(f"{len(vessels)} unique vessels")
    vessels.select("mmsi", "vessel_name", "flag", "ship_type").head(10)

## Fishing Effort (4Wings Grid)

The 4Wings API provides **aggregated** fishing effort on a spatial grid.
Each row is a grid cell with total vessel-hours, broken down by flag
state and gear type.

This is fundamentally different from position data — there are no
individual vessel tracks, just aggregate activity per cell.

| Column | Description |
|---|---|
| `date` | Activity date |
| `lat`, `lon` | Grid cell center |
| `flag` | Flag state ISO code |
| `geartype` | Fishing gear category |
| `vessel_hours` | Total fishing hours in the cell |

In [ ]:
if token:
    effort = n.fishing_effort().collect()
    print(f"{len(effort)} grid cells")
    print(f"Total vessel-hours: {effort['vessel_hours'].sum():.1f}")
    print()
    print("Top flags by vessel-hours:")
    print(
        effort.group_by("flag")
        .agg(hours=pl.col("vessel_hours").sum())
        .sort("hours", descending=True)
        .head(5)
    )
    print()
    effort.head(5)

In [ ]:
if token:
    print("Effort by gear type:")
    print(
        effort.group_by("geartype")
        .agg(hours=pl.col("vessel_hours").sum(), cells=pl.len())
        .sort("hours", descending=True)
        .head(10)
    )

## (Optional) Enrich NOAA Positions with GFW Events

GFW events and NOAA positions are complementary — you can join them
by MMSI and time window to see which vessels in your position data
had GFW-detected fishing activity.

This is **cross-source enrichment**, not position fusion (which is
covered in [Example 03](03_multi_source_fusion.ipynb)).

In [ ]:
# Uncomment to run — requires NOAA data from Example 02
#
# n_noaa = Neptune("2024-06-15", sources=["noaa"], cache_dir="/tmp/neptune_demo")
# n_noaa.download()
# positions = n_noaa.positions().collect()
#
# if token:
#     fishing = n.events(kind="fishing").collect()
#     # Find vessels that appear in both datasets
#     gfw_mmsis = fishing["mmsi"].unique()
#     matched = positions.filter(pl.col("mmsi").is_in(gfw_mmsis))
#     print(f"{len(gfw_mmsis)} vessels with GFW fishing events")
#     print(f"{matched['mmsi'].n_unique()} of those found in NOAA positions")
#     print(f"{len(matched)} position reports for matched vessels")

## Next Steps

- **[Example 04 — Event Detection](04_event_detection.ipynb)**: Derive events from raw
  positions using Neptune's heuristic detectors (complementary to GFW's source-native events)
- **[Example 06 — External Plugin](06_external_plugin.py)**: Build a custom adapter to
  integrate additional data sources